# 02 - Feature Quality Checks
**DNA Gene Mapping Project - ML Phase V5 - Feature Selection**  
**Author:** Sharique Mohammad  
**Date:** February 2026

## Purpose
For every use case, measure the quality of each candidate feature:
- Missing value rate
- Zero-variance detection
- Data type confirmation

Any column with more than 50% missing values or zero variance is flagged for removal.
Results feed into 05_feature_reduction.ipynb.

## Output
`data/feature_selection/02_feature_quality.csv`
  - use_case, table, column, dtype, missing_pct, zero_variance, keep

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

PROJECT_ROOT = Path().absolute().parent.parent
FS_DIR       = PROJECT_ROOT / 'data' / 'feature_selection'
FS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Setup complete")
print(f"Feature selection dir: {FS_DIR}")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)
print(f"Connected: {POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")

## 3. Use Case Registry

In [ ]:
USE_CASES = [
    # (uc_code, table, target, target_type, sample_pct, cv_required)
    ('UC01', 'clinical_ml_features',               'target_is_pathogenic',                 'binary',     10,  False),
    ('UC02', 'disease_ml_features',                'is_pathogenic',                        'binary',     10,  False),
    ('UC03', 'pharmacogene_ml_features',           'is_pathogenic',                        'binary',     10,  False),
    ('UC04', 'variant_impact_ml_features',         'is_high_impact',                       'binary',     10,  False),
    ('UC05', 'structural_variant_ml_features',     'sv_classification',                    'multiclass', 100, False),
    ('UC06', 'variant_drug_response_ml_features',  'is_actionable_pharmacogene_variant',   'binary',     10,  False),
    ('UC07', 'variant_cancer_ml_features',         'is_driver_candidate',                  'binary',     10,  False),
    ('UC08', 'variant_population_ml_features',     'is_carrier_screening_candidate',       'binary',     100, False),
    ('UC09', 'population_frequency_ml_features',   'is_clinically_actionable_rare_variant','binary',     100, False),
    ('UC10', 'gene_pharmacogene_ml_features',      'is_high_priority_pharmacogene',        'binary',     100, True),
    ('UC11', 'gene_expression_ml_features',        'is_clinically_relevant_expression',    'binary',     100, False),
    ('UC12', 'gene_protein_family_ml_features',    'is_high_value_protein_family',         'binary',     100, False),
    ('UC13', 'gene_test_availability_ml_features', 'is_high_priority_test_gene',           'binary',     100, False),
    ('UC14', 'transcript_expression_ml_features',  'is_clinically_relevant_expression',    'binary',     100, False),
    ('UC15', 'cancer_variant_ml_features',         'gene_cancer_role',                     'multiclass', 10,  False),
]

def load_sample(table, engine, pct):
    if pct == 100:
        return pd.read_sql(f"SELECT * FROM gold.{table}", engine)
    return pd.read_sql(f"SELECT * FROM gold.{table} TABLESAMPLE SYSTEM ({pct})", engine)

def fix_bool(series):
    return series.astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False})

def fix_types(df, schema_df, table):
    tbl = schema_df[schema_df['table_name'] == table]
    for _, row in tbl.iterrows():
        c, dt = row['column_name'], row['data_type']
        if c not in df.columns:
            continue
        if dt in ('INT', 'BIGINT'):
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
        elif dt == 'DOUBLE':
            df[c] = pd.to_numeric(df[c], errors='coerce')
        elif dt == 'BOOLEAN':
            df[c] = fix_bool(df[c])
    return df

print(f"Registry loaded: {len(USE_CASES)} use cases")

## 4. Quality Check Loop

In [ ]:
# ID/metadata columns - never features regardless of quality
ID_COLS = {
    'variant_id', 'sv_id', 'gene_symbol', 'gene_name', 'gene_full_name',
    'official_gene_symbol', 'official_symbol', 'validated_gene_symbol',
    'pharmgkb_name', 'chromosome', 'position', 'study_id', 'variant_name',
    'variant_key', 'assembly', 'reference_allele', 'alternate_allele',
    'protein_name', 'refseq_protein_accession', 'uniprot_accession',
    'omim_id', 'mondo_id', 'orphanet_id', 'description',
    'cdna_change', 'protein_change', 'start_pos', 'end_pos',
    'variant_pharmgkb_id', 'gene_list'
}

MISSING_THRESHOLD = 0.50

all_rows = []

for uc, table, target, ttype, pct, cv in USE_CASES:
    print(f"--- {uc} | {table} ---")
    df = load_sample(table, engine, pct)
    n  = len(df)
    flagged_missing = 0
    flagged_zv      = 0

    for col in df.columns:
        if col == target:
            continue
        is_id = col in ID_COLS

        miss_rate  = df[col].isnull().mean()
        high_miss  = miss_rate > MISSING_THRESHOLD

        numeric    = pd.to_numeric(df[col], errors='coerce')
        zero_var   = numeric.nunique(dropna=True) <= 1 if df[col].dtype != object else df[col].nunique(dropna=True) <= 1

        keep = not is_id and not high_miss and not zero_var

        if not keep:
            if is_id:        reason = 'id_col'
            elif high_miss:  reason = f'missing_{miss_rate*100:.0f}pct'; flagged_missing += 1
            else:            reason = 'zero_variance'; flagged_zv += 1
        else:
            reason = ''

        all_rows.append({
            'use_case': uc, 'table': table, 'column': col,
            'dtype': str(df[col].dtype), 'missing_pct': round(miss_rate * 100, 2),
            'zero_variance': zero_var, 'is_id_col': is_id,
            'keep': keep, 'remove_reason': reason
        })

    total_cols    = len(df.columns) - 1
    removed_id    = sum(1 for r in all_rows if r['use_case'] == uc and r['is_id_col'])
    removed_total = sum(1 for r in all_rows if r['use_case'] == uc and not r['keep'])
    kept_total    = sum(1 for r in all_rows if r['use_case'] == uc and r['keep'])
    print(f"  cols={total_cols}  removed_id={removed_id}  removed_missing={flagged_missing}  removed_zv={flagged_zv}  kept={kept_total}")
    print()

## 5. Save Quality Report

In [ ]:
quality_df = pd.DataFrame(all_rows)
quality_df.to_csv(FS_DIR / '02_feature_quality.csv', index=False)
print("Saved: 02_feature_quality.csv")
print()

# Summary per use case
summary = quality_df.groupby('use_case').agg(
    total_cols  = ('column', 'count'),
    removed_id  = ('is_id_col', 'sum'),
    high_missing= ('missing_pct', lambda x: (x > 50).sum()),
    zero_var    = ('zero_variance', 'sum'),
    kept        = ('keep', 'sum')
).reset_index()
print(summary.to_string(index=False))

## 6. Visualize Missing Value Rates

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, (uc, table, target, ttype, pct, cv) in enumerate(USE_CASES):
    uc_df = quality_df[(quality_df['use_case'] == uc) & (quality_df['keep'] == True)]
    miss  = uc_df['missing_pct'].sort_values(ascending=False).head(20)

    if len(miss) > 0:
        axes[i].barh(range(len(miss)), miss.values, color='steelblue', alpha=0.8)
        axes[i].set_yticks(range(len(miss)))
        axes[i].set_yticklabels(miss.index.tolist() if isinstance(miss.index[0], str)
                                  else uc_df.loc[miss.index, 'column'].tolist(), fontsize=6)
        axes[i].axvline(50, color='red', linestyle='--', linewidth=1)
    axes[i].set_title(f"{uc}", fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Missing %', fontsize=7)
    axes[i].grid(axis='x', alpha=0.3)

plt.suptitle('Top Missing Value Rates per Use Case (red = 50% threshold)', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig(FS_DIR / '02_missing_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_missing_rates.png")

## 7. Summary

In [ ]:
print("=" * 55)
print("FEATURE QUALITY CHECKS COMPLETE")
print("=" * 55)
total_kept    = (quality_df['keep'] == True).sum()
total_removed = (quality_df['keep'] == False).sum()
print(f"Total feature-use case pairs : {len(quality_df)}")
print(f"Kept after quality filter    : {total_kept}")
print(f"Removed                      : {total_removed}")
print(f"  - ID/metadata columns      : {quality_df['is_id_col'].sum()}")
print(f"  - High missing (>50%)      : {(quality_df['missing_pct'] > 50).sum()}")
print(f"  - Zero variance            : {quality_df['zero_variance'].sum()}")
print()
print("Next: 03_correlation_analysis.ipynb")